# Capacity test — single-seed deep-big pilot

This notebook runs one pre-registered seed-0 screening trial of the 737,844-parameter deep-big class-conditional MoE. It preserves the Notebook 43 incumbent training method and changes only encoder, latent, expert, and gate capacity. The pilot is restart-safe and does not justify a final capacity claim without a later paired multi-seed confirmation.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
REFERENCE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_class_conditional_deeper_gate32_frozen_stagec_v1'
REFERENCE_CONFIG = 'config/compact_soft_moe_balanced_supcon_class_conditional_frozen_deeper_gate_3seed.yaml'
CAPACITY_PREFIX = 'nfv3_4way_moe_capacity_deep_big_v1'
CAPACITY_CONFIG = 'config/capacity_test_deep_big_3seed.yaml'
PILOT_SEED = 0  # Locked. This notebook must not choose a favorable seed.
EXECUTE = False  # Run the dry-run first; then change to True.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

The GitHub token is supplied only through an ephemeral Git header and is never embedded in the clone URL or printed. Checkpoints, logs, and reports are stored on Drive so a disconnected Colab session can resume safely.

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Locked contract, topology preflight, and seed-0 execution

The checks below prove the exact topology, parameter counts, frozen Stage-C policy, and allowlisted difference from the 22,420-parameter reference. The capacity run starts from fresh Stage A/B/C parameters; no small-model checkpoint is reused. Both dry-run and execution commands are hard-locked to seed 0 and at most one new seed transaction.

In [ ]:
import copy, torch, yaml
from IPython.display import display
from models.encoder import build_encoder, resolve_encoder_config
from training.compact_soft_moe_run import CompactSoftMoEStudy
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if PILOT_SEED != 0: raise ValueError('The registered pilot seed is 0.')
if EXECUTE and not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before setting EXECUTE=True.')
reference = yaml.safe_load(Path(REFERENCE_CONFIG).read_text())
capacity = yaml.safe_load(Path(CAPACITY_CONFIG).read_text())
assert reference['seeds'] == capacity['seeds'] == [0, 1, 2]
assert 'reuse_stage_a_from_prefix' not in capacity and 'reuse_stage_b_from_prefix' not in capacity
ref_contract = copy.deepcopy(reference['backbone'])
capacity_contract = copy.deepcopy(capacity['backbone'])
assert ref_contract['architecture'] == capacity_contract['architecture'] == 'moe_dataset_class_conditional'
for key in ('latent_dim',): capacity_contract['model'][key] = ref_contract['model'][key]
for component, key in (('encoder', 'hidden_dims'), ('expert', 'hidden_dims'), ('gate', 'hidden_dims')):
    capacity_contract['model'][component][key] = ref_contract['model'][component][key]
assert capacity_contract == ref_contract, 'Only encoder/latent/expert/gate capacity may differ from the reference'
model_cfg = capacity['backbone']['model']
assert model_cfg['encoder']['hidden_dims'] == [512, 512]
assert model_cfg['latent_dim'] == 256
assert model_cfg['expert']['hidden_dims'] == [256]
assert model_cfg['gate']['hidden_dims'] == [128]
encoder = build_encoder(47, 256, resolve_encoder_config(model_cfg))
model = build_model('moe_dataset_class_conditional', encoder, ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], model_cfg)
apply_stage_c_trainability(model, capacity['backbone'])
assert model.class_reliability.shape == (4, 22)
assert torch.count_nonzero(model.class_reliability).item() == 0
assert sum(p.numel() for p in model.parameters()) == capacity['expected_total_parameters'] == 737844
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 33500
runner = CompactSoftMoEStudy(base_config_path='config/default.yaml', study_config_path=CAPACITY_CONFIG, prefix=CAPACITY_PREFIX, execute=False)
nested = runner._final_nested(PILOT_SEED)
assert nested['seed'] == nested['data']['split_seed'] == PILOT_SEED
assert nested['training']['stages'] == ['A', 'B', 'C']
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_capacity_test.py', 'tests/test_dataset_blind_inference.py', 'tests/test_compact_soft_moe_run.py'], check=True)
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CAPACITY_PREFIX}_summary'
summary_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', CAPACITY_CONFIG, '--prefix', CAPACITY_PREFIX, '--only-seed', str(PILOT_SEED), '--max-new-seeds', '1']
run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label='capacity-test-seed0-dry-run')
if EXECUTE:
    live_runner = CompactSoftMoEStudy(base_config_path='config/default.yaml', study_config_path=CAPACITY_CONFIG, prefix=CAPACITY_PREFIX, execute=True, max_new_seeds=1)
    print('Restart audit:', live_runner.audit_seed(PILOT_SEED))
    run_streaming_logged([*command, '--execute'], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label='capacity-test-seed0-training')
else:
    print('NO TRAINING WAS STARTED. Review the assertions, set EXECUTE=True, and rerun this cell.')

## Seed-0 capacity comparison

The candidate is compared with the same seed from the Notebook 43 incumbent. The test is a screening result: practical advancement requires at least +0.003 macro-F1, no more than 0.001 accuracy loss, and no dataset macro-F1 loss greater than 0.01. Missing reference artifacts are reported rather than silently replaced by another baseline.

In [ ]:
import numpy as np, pandas as pd
candidate_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CAPACITY_PREFIX}_seed{PILOT_SEED}'
reference_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{REFERENCE_PREFIX}_seed{PILOT_SEED}'
required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Per_Class_Metrics.csv', 'Resource_Accounting.csv']
if not all((candidate_dir / name).is_file() for name in required):
    print('Seed-0 capacity run is not complete yet:', candidate_dir)
else:
    candidate_resource = pd.read_csv(candidate_dir / 'Resource_Accounting.csv')
    assert set(candidate_resource['total_parameters'].astype(int)) == {737844}
    assert set(candidate_resource['trainable_parameters'].astype(int)) == {33500}
    print('=== Capacity resource accounting ==='); display(candidate_resource)
    candidate_overall = pd.read_csv(candidate_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").iloc[0]
    if not all((reference_dir / name).is_file() for name in required):
        print('Capacity result exists, but the registered seed-0 reference is missing:', reference_dir)
        display(candidate_overall.to_frame('capacity_seed0'))
    else:
        reference_overall = pd.read_csv(reference_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").iloc[0]
        metric_names = [name for name in ('macro_f1', 'accuracy', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro') if name in candidate_overall.index and name in reference_overall.index]
        headline = pd.DataFrame({'reference_22420': reference_overall[metric_names].astype(float), 'capacity_737844': candidate_overall[metric_names].astype(float)})
        headline['delta'] = headline['capacity_737844'] - headline['reference_22420']
        print('=== Locked-test headline metrics ==='); display(headline)
        ref_ds = pd.read_csv(reference_dir / 'Per_Dataset_Metrics.csv')[['dataset', 'macro_f1']].rename(columns={'macro_f1': 'reference_macro_f1'})
        cap_ds = pd.read_csv(candidate_dir / 'Per_Dataset_Metrics.csv')[['dataset', 'macro_f1']].rename(columns={'macro_f1': 'capacity_macro_f1'})
        dataset_delta = ref_ds.merge(cap_ds, on='dataset', validate='one_to_one')
        dataset_delta['delta'] = dataset_delta['capacity_macro_f1'] - dataset_delta['reference_macro_f1']
        print('=== Per-dataset macro-F1 ==='); display(dataset_delta)
        rare_names = ['Reconnaissance', 'Theft', 'Shellcode', 'Worms', 'WebAttacks', 'Analysis']
        ref_class = pd.read_csv(reference_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL' and `class` in @rare_names")[['class', 'support', 'recall', 'f1']].rename(columns={'recall': 'reference_recall', 'f1': 'reference_f1'})
        cap_class = pd.read_csv(candidate_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL' and `class` in @rare_names")[['class', 'recall', 'f1']].rename(columns={'recall': 'capacity_recall', 'f1': 'capacity_f1'})
        rare_delta = ref_class.merge(cap_class, on='class', validate='one_to_one')
        rare_delta['recall_delta'] = rare_delta['capacity_recall'] - rare_delta['reference_recall']
        rare_delta['f1_delta'] = rare_delta['capacity_f1'] - rare_delta['reference_f1']
        print('=== Named rare/collapsed classes ==='); display(rare_delta)
        macro_delta = float(candidate_overall['macro_f1'] - reference_overall['macro_f1'])
        accuracy_delta = float(candidate_overall['accuracy'] - reference_overall['accuracy'])
        worst_dataset_delta = float(dataset_delta['delta'].min())
        advance = macro_delta >= 0.003 and accuracy_delta >= -0.001 and worst_dataset_delta >= -0.01
        print({'macro_f1_delta': macro_delta, 'accuracy_delta': accuracy_delta, 'worst_dataset_delta': worst_dataset_delta, 'advance_to_three_seed_confirmation': advance})

## Routing, reliability, and convergence diagnostics

In [ ]:
import matplotlib.pyplot as plt
if not (candidate_dir / 'manifest.json').is_file():
    print('Diagnostics will appear after the capacity seed completes.')
else:
    checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{CAPACITY_PREFIX}_seed{PILOT_SEED}'
    reports = [('Gate_By_Dataset.csv', 'Gate weights by source dataset', candidate_dir), ('Expert_Utilization.csv', 'Expert utilization', candidate_dir), ('Class_Conditional_Reliability.csv', 'Learned expert × class reliability', candidate_dir), ('Validation_Expert_Owned_StageB.csv', 'Owned-expert validation after Stage B', checkpoint_dir), ('Validation_Expert_Owned_StageC.csv', 'Owned-expert validation after Stage C', candidate_dir)]
    for filename, title, directory in reports:
        path = directory / filename
        if path.is_file(): print(f'=== {title} ==='); display(pd.read_csv(path))
    history_path = candidate_dir / 'training' / 'Training_History.csv'
    if history_path.is_file():
        history = pd.read_csv(history_path)
        columns = [c for c in ['stage', 'epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_metric_loss', 'train_balance_penalty', 'train_reliability_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if c in history]
        display(history[columns])
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for stage, frame in history.groupby('stage'):
            if 'train_total_loss' in frame and frame['train_total_loss'].notna().any(): axes[0].plot(frame['epoch'], frame['train_total_loss'], marker='o', label=f'Stage {stage}')
        stage_c = history.query("stage == 'C'")
        if 'val_macro_f1' in stage_c and stage_c['val_macro_f1'].notna().any(): axes[1].plot(stage_c['epoch'], stage_c['val_macro_f1'], marker='o', label='validation macro-F1')
        axes[0].set(title='Training loss', xlabel='epoch'); axes[1].set(title='Stage C convergence', xlabel='epoch')
        for axis in axes: axis.grid(alpha=0.25); axis.legend()
        plt.tight_layout(); plt.show()

## Interpretation

This is a single-seed screen, not a definitive capacity result. Meeting all three practical guardrails supports spending compute on the locked three-seed confirmation. A loss means only that this 737,844-parameter deep-big model did not improve seed 0 under the unchanged `1e-3`, `30/10/30` training schedule; it cannot establish that larger capacity is universally ineffective.